# Metody algorytmiczne — porównanie podejść
**Ćwiczenia do wykładu (45 min)** · wersja do uzupełnienia

Cel: na *tych samych danych* porównać cztery podejścia z wykładu:
1. model nadzorowany: regresja logistyczna (z wagami klas),
2. model nadzorowany: Random Forest (z wagami klas),
3. detektor anomalii: Isolation Forest,
4. detektor anomalii: PCA.

Na końcu (Część 6 - dla chętnych) wypróbujesz ideę anomalii na **symulowanych danych rynkowych**.
Zadania oznaczone `TODO` uzupełniasz samodzielnie. Czas: ok. 45–60 minut.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Część 1. Dane i punkt odniesienia
Generujemy syntetyczny zbiór „transakcji” z ok. 1% oszustw (`make_classification`).
Dane są sztuczne — służą porównaniu metod, nie wnioskom o prawdziwych oszustwach.

In [ ]:
X, y = make_classification(n_samples=20000, n_features=10, n_informative=6, n_redundant=2,
                           n_clusters_per_class=2, weights=[0.99, 0.01], flip_y=0.0,
                           class_sep=1.2, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
print(f"Trening: {len(y_train)} wierszy, oszustw: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test:    {len(y_test)} wierszy, oszustw: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

### Zadanie 1 — pułapka trafności
Oblicz trafność (accuracy), precision i recall modelu, który **zawsze** przewiduje „legalna” (klasa 0).

In [ ]:
y_pred_trivial = None  # TODO
acc = None             # TODO 
rec = None             # TODO
print(f"accuracy = {acc}, recall = {rec}")

Pomocnicza funkcja do oceny — dla wyniku ciągłego `score` (większy = bardziej podejrzane) liczy PR-AUC oraz metryki przy zadanym progu.

In [ ]:
def ocen(nazwa, score, prog):
    pred = (score >= próg).astype(int)
    return {"metoda": nazwa,
            "precision": round(precision_score(y_test, pred, zero_division=0), 3),
            "recall": round(recall_score(y_test, pred, zero_division=0), 3),
            "f1": round(f1_score(y_test, pred, zero_division=0), 3),
            "pr_auc": round(average_precision_score(y_test, score), 3),
            "alerty": int(pred.sum())}
wyniki = []

## Część 2. Modele nadzorowane (poziom algorytmu)
### Zadanie 2 — regresja logistyczna z wagami klas
Wytrenuj `LogisticRegression` z `class_weight="balanced"`. Wynik: prawdopodobieństwo klasy 1. Użyj progu 0,5.

In [ ]:
lr = None  # TODO
score_lr = None  # TODO
wyniki.append(oceń("Regresja logistyczna (wagi)", score_lr, 0.5))
pd.DataFrame(wyniki)

### Zadanie 3 — Random Forest z wagami klas

In [ ]:
rf = None  # TODO
score_rf = None  # TODO
wyniki.append(oceń("Random Forest (wagi)", score_rf, 0.5))
pd.DataFrame(wyniki)

### Zadanie 4 — próg z kosztów
Załóż: przeoczone oszustwo (FN) kosztuje **500 zł**, fałszywy alarm (FP) — **10 zł**.
1. Oblicz próg teoretyczny `C_FP / (C_FP + C_FN)`.
2. Policz łączny koszt Random Forest przy progu 0,5 i przy progu teoretycznym.
3. Zastanów się: czy próg teoretyczny jest tu wiarygodny, skoro użyliśmy wag klas?

In [ ]:
C_FP, C_FN = 10, 500
prog_teor = None  # TODO

def koszt(score, prog):
    pred = (score >= prog).astype(int)
    fp = None  # TODO
    fn = None  # TODO
    return fp * C_FP + fn * C_FN

print("próg teoretyczny:", prog_teor)
print("koszt przy 0.5:", koszt(score_rf, 0.5))
print("koszt przy progu teoretycznym:", koszt(score_rf, prog_teor))

## Część 3. Detektory anomalii — uczą się „normy”
Modele z tej części **nie widzą etykiet oszustw w treningu**. Etykiet użyjemy wyłącznie do oceny.
Próg wyznaczamy tak, by liczba alertów odpowiadała budżetowi analityków: oznaczamy **1,5%** najbardziej podejrzanych transakcji testowych.

In [ ]:
BUDZET = 0.015
def prog_z_budzetu(score, udział=BUDZET):
    return np.quantile(score, 1 - udzial)

### Zadanie 5 — Isolation Forest
Wytrenuj las izolacyjny na **całym** zbiorze treningowym (bez etykiet). W scikit-learn `score_samples` zwraca wartości *większe = bardziej normalne*, więc wynik anomalii to `-score_samples`.

In [ ]:
iso = None  # TODO: IsolationForest(n_estimators=200, random_state=RANDOM_STATE).fit(Xtr)
score_if = None  # TODO: -iso.score_samples(Xte)
wyniki.append(ocen("Isolation Forest", score_if, prog_z_budzetu(score_if)))
pd.DataFrame(wyniki)

### Zadanie 6 — PCA i autoenkoder: błąd rekonstrukcji
Trenujemy **tylko na transakcjach legalnych** ze zbioru treningowego (`Xtr[y_train == 0]`).
- PCA: rzut na `k = 3` składowe i odtworzenie (`inverse_transform`).

Wynik anomalii = błąd rekonstrukcji `‖x − x'‖²` (suma kwadratów różnic po cechach).

In [ ]:
Xnorm = Xtr[y_train == 0]

pca = None  # TODO
rec_pca = None  # TODO
score_pca = None  # TODO

wyniki.append(ocen("PCA (rekonstrukcja)", score_pca, prog_z_budzetu(score_pca)))
pd.DataFrame(wyniki)

## Część 4. Porównanie na jednej skali
Modele nadzorowane użyły progu 0,5, a detektory anomalii — progu z budżetu alertów, więc kolumny precision/recall nie są w pełni porównywalne. **PR-AUC** nie zależy od progu — to uczciwe porównanie jakości rankingu.

In [ ]:
tabela = pd.DataFrame(wyniki).sort_values("pr_auc", ascending=False)
display(tabela)

scores = {"Regresja logistyczna": score_lr, "Random Forest": score_rf, "Isolation Forest": score_if, "PCA": score_pca}
fig, ax = plt.subplots(figsize=(7, 5))
for nazwa, sc in scores.items():
    pass  # TODO: p, r, _ = precision_recall_curve(y_test, sc); ax.plot(r, p, label=nazwa)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(); ax.set_title("Krzywe precision–recall")
plt.show()

## Część 5. Eksperyment: co, gdy etykiet prawie nie ma?
Zasymulujmy sytuację „zimnego startu”: model nadzorowany dostaje tylko **10 oszustw** w treningu (reszta danych bez zmian). Detektory anomalii się nie zmieniają — nie używały etykiet.

### Zadanie 7
Wytrenuj Random Forest (z wagami klas) na zbiorze, w którym zostawiono losowo 10 oszustw i wszystkie legalne transakcje. Porównaj jego PR-AUC z Isolation Forest z Zadania 5.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
idx_fraud = np.where(y_train == 1)[0]
idx_legit = np.where(y_train == 0)[0]
zostaw = None  # TODO: np.concatenate([idx_legit, rng.choice(idx_fraud, 10, replace=False)])

rf_malo = None  # TODO: RandomForestClassifier(... class_weight="balanced_subsample" ...).fit(Xtr[zostaw], y_train[zostaw])
score_rf_malo = None  # TODO: predict_proba(Xte)[:, 1]
print("PR-AUC RF (10 oszustw):", round(average_precision_score(y_test, score_rf_mało), 3))
print("PR-AUC Isolation Forest:", round(average_precision_score(y_test, score_if), 3))

## Część 6. Anomalie rynkowe (symulacja)
Zamiast transakcji — **dni obrotu jednego instrumentu**. Symulujemy 750 dni (ok. 3 lata): zwroty ze zmienną zmiennością, wolumen skorelowany ze zmiennością, oraz **12 wstrzykniętych zdarzeń** (np. nagły skok wolumenu i nietypowy zwrot bez zmiany zmienności). To dane sztuczne — znamy „prawdę”, więc możemy ocenić detektor.

Idea z wykładu: zdefiniuj normę w **ruchomym oknie** i opisz zdarzenie cechami *względnymi*.

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
n = 750
vol = 0.01 * np.exp(0.5 * np.cumsum(rng.normal(0, 0.05, n)))    # wolno zmieniająca się zmienność
ret = rng.normal(0, vol)
volume = np.exp(np.log(1e6) + 15 * (vol - vol.mean()) / vol.std() * 0.1 + rng.normal(0, 0.15, n))
etykieta = np.zeros(n, dtype=int)
zdarzenia = rng.choice(np.arange(60, n), 12, replace=False)
ret[zdarzenia] += rng.choice([-1, 1], 12) * 4 * vol[zdarzenia]   # nietypowy zwrot
volume[zdarzenia] *= rng.uniform(2.5, 4.0, 12)                    # skok wolumenu
etykieta[zdarzenia] = 1
rynek = pd.DataFrame({"zwrot": ret, "wolumen": volume, "zdarzenie": etykieta})
print("dni:", n, " wstrzykniętych zdarzeń:", etykieta.sum())
rynek.head()

### Zadanie 8 — cechy względne w ruchomym oknie
Dla okna `W = 60` dni (tylko **przeszłość**, bez bieżącego dnia — `.shift(1)`) policz:
- `z_zwrot` = zwrot / odchylenie standardowe zwrotów w oknie,
- `z_wolumen` = (log wolumenu − średnia log wolumenu w oknie) / odchylenie standardowe w oknie.

Dlaczego `shift(1)`? Gdyby okno zawierało bieżący dzień, anomalia „rozmywałaby” własny punkt odniesienia.

In [ ]:
W = 60
lw = np.log(rynek["wolumen"])
sd_ret = None  # TODO: rynek["zwrot"].rolling(W).std().shift(1)
mu_lw  = None  # TODO: lw.rolling(W).mean().shift(1)
sd_lw  = None  # TODO: lw.rolling(W).std().shift(1)
rynek["z_zwrot"] = None    # TODO
rynek["z_wolumen"] = None  # TODO
cechy = rynek.dropna().copy()
print(len(cechy), "dni z kompletnymi cechami; zdarzeń:", cechy["zdarzenie"].sum())

### Zadanie 9 — dwie proste metody i Isolation Forest
Porównaj trzy wyniki anomalii na `cechy`:
1. **reguła z-score**: `max(|z_zwrot|, |z_wolumen|)`,
2. **Isolation Forest** na dwóch cechach (`z_zwrot`, `z_wolumen`),
3. **surowy zwrot bez normalizacji**: `|zwrot|` (punkt odniesienia — czy normalizacja do reżimu zmienności pomaga?).

Oceń PR-AUC każdej metody względem kolumny `zdarzenie`.

In [ ]:
F = cechy[["z_zwrot", "z_wolumen"]].values
yy = cechy["zdarzenie"].values

s_z = None    # TODO: np.maximum(np.abs(F[:, 0]), np.abs(F[:, 1]))
s_if = None   # TODO: -IsolationForest(n_estimators=200, random_state=RANDOM_STATE).fit(F).score_samples(F)
s_raw = None  # TODO: np.abs(cechy["zwrot"].values)

for nazwa, sc in [("z-score", s_z), ("Isolation Forest", s_if), ("surowy |zwrot|", s_raw)]:
    print(f"{nazwa:18s} PR-AUC = {average_precision_score(yy, sc):.3f}")

### Zadanie 10 — alerty w czasie
Zaznacz na wykresie zwrotów dni, które Isolation Forest oznaczył jako **top 15** (najwyższy wynik), oraz prawdziwe zdarzenia. Ile z 15 alertów to prawdziwe zdarzenia?

In [ ]:
top = None  # TODO: indeksy (pozycje w cechy) 15 największych wartości s_if → np.argsort(s_if)[-15:]
trafione = None  # TODO: yy[top].sum()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(cechy.index, cechy["zwrot"], lw=0.6, color="gray")
# TODO: ax.scatter(cechy.index[top], cechy["zwrot"].values[top], color="crimson", label="alert")
# TODO: ax.scatter(cechy.index[yy == 1], cechy["zwrot"].values[yy == 1], facecolors="none", edgecolors="navy", s=90, label="prawdziwe zdarzenie")
ax.legend(); ax.set_title("Alerty Isolation Forest vs prawdziwe zdarzenia")
plt.show()
print("trafione:", trafione, "z 15")

## Pytania do przemyślenia
1. Który model ma najwyższy PR-AUC na danych z pełnymi etykietami? Czy detektory anomalii mogą z nim konkurować?
2. Co się zmienia w Zadaniu 7, gdy etykiet jest mało? Jaki wniosek wynika dla „zimnego startu”?
3. Czy normalizacja do ruchomego okna pomogła w Części 6? Od czego zależy, kiedy cechy względne dają przewagę nad surowym zwrotem (podpowiedź: jak bardzo zmienia się zmienność w danych)?
4. Podaj jedno zdarzenie, które detektor anomalii oznaczy jako podejrzane, a które nie będzie przestępstwem.